In [ ]:
!pip install tensorflow -q

# AI-Based Vehicle Crash Detection System

## Objective
This project detects vehicle crash sounds using transfer learning with Google's YAMNet model.

Instead of training directly on raw audio, audio clips are first converted into 1024-dimensional feature embeddings using YAMNet. These embeddings are then used to train a binary classifier that distinguishes crash sounds from normal background sounds.

The final trained model is exported to TensorFlow Lite (.tflite) format for deployment on edge devices and mobile systems.

---

## Import required libraries:

In [ ]:
import tensorflow as tf
import numpy as np

# Loading precomputed YAMNet embeddings and labels:

**X_embeddings.npy:**

Contains 1024-dimensional feature vectors extracted from audio clips

**y_labels.npy:**

Binary labels:
- 1 = Crash sound
- 0 = Background / Non-crash sound

In [ ]:
X = np.load("X_embeddings.npy")
y = np.load("y_labels.npy")

print(X.shape)
print(y.shape)

(612, 1024)
(612,)


# Dataset Preparation:

The dataset has already been processed using YAMNet.

Each audio clip was converted into a 1024-dimensional embedding vector. These embeddings are used as inputs for classifier training.

In [ ]:
from sklearn.model_selection import train_test_split

## Splitting the dataset into training and testing sets:
- 80% Training
- 20% Testing

**Note:** Stratified splitting preserves class balance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape)
print(X_test.shape)

(489, 1024)
(123, 1024)


## Classifier Architecture:

A fully connected neural network is trained on top of YAMNet embeddings.

Architecture:

- Input Layer (1024 features)
- Dense Layer (256 neurons)
- Dropout
- Dense Layer (64 neurons)
- Dropout
- Output Layer (Binary Classification)

In [ ]:
# Define neural network classifier:

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1024,)),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

In [ ]:
# Compile model using Adam optimizer
# Binary Cross Entropy is used because this is a binary classification task.

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 278,913 (1.06 MB)

 Trainable params: 278,913 (1.06 MB)

 Non-trainable params: 0 (0.00 B)

## Model Training:

The classifier is trained using the extracted embeddings.

Training Parameters:

- Epochs: 20
- Batch Size: 32
- Optimizer: Adam
- Loss Function: Binary Crossentropy

In [ ]:
# Training the classifier:

history = model.fit(

    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=32
)

Epoch 1/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.6871 - loss: 0.6640 - val_accuracy: 0.8455 - val_loss: 0.3812
Epoch 2/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8855 - loss: 0.4475 - val_accuracy: 0.9187 - val_loss: 0.2459
Epoch 3/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.9243 - loss: 0.2934 - val_accuracy: 0.9187 - val_loss: 0.2097
Epoch 4/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9141 - loss: 0.2748 - val_accuracy: 0.9268 - val_loss: 0.1727
Epoch 5/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.9264 - loss: 0.2717 - val_accuracy: 0.9350 - val_loss: 0.1905
Epoch 6/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.9489 - loss: 0.1563 - val_accuracy: 0.9431 - val_loss: 0.1586
Epoch 7/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.9652 - loss: 0.1355 - val_accuracy: 0.9512 - val_loss: 0.1171
Epoch 8/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - accuracy: 0.9652 - loss: 0.1371 - val_accuracy: 0.9431 - v

## Model Evaluation:

After training, model performance is evaluated on the test dataset.

In [ ]:
# Evaluate model performance

loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy:.4f}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9756 - loss: 0.1470
Test Accuracy: 0.9756


## Saving Trained Model:

The trained neural network is saved in HDF5 format for future use.

In [ ]:
model.save("crash_detector_model.h5")

print("Model saved!!!")

✅ Model saved.


## TensorFlow Lite Conversion:

The trained model is converted to TensorFlow Lite format for deployment on embedded systems and mobile devices.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model = converter.convert()

with open("crash_detector_model.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model exported!!!")

Saved artifact at '/tmp/tmpt5m5huxj'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1024), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  138207424204240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138207424205584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138207424206928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138207424205200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138207424205008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138207424207120: TensorSpec(shape=(), dtype=tf.resource, name=None)
✅ TFLite model exported.


## Downloading Trained Model:

Download the generated TensorFlow Lite model.

In [ ]:
from google.colab import files

files.download("crash_detector_model.tflite")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Conclusion:

This project demonstrates a lightweight accident detection pipeline using transfer learning.

Rather than training a deep audio model from scratch, YAMNet embeddings were used as a compact representation of audio events. A custom classifier was then trained to distinguish crash sounds from normal road sounds.

The final model was exported as a TFLite model, making it suitable for deployment on resource-constrained devices.